# Continuous Wavelet Transform (CWT)

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

The Continuous Wavelet Transform decomposes the signal into translated and scaled functions, enabling a time-frequency analysis that tracks how frequencies change over time. We use the complex Morlet wavelet (cmor1.5-1.0) from the PyWavelets library.

## Expected outputs

- A heatmap (scalogram) showing the energy at each frequency and each moment
- High-energy regions appearing and disappearing at specific times
- Unlike Fourier transform, this analysis reveals when each frequency appeared

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| Frequencies | 0.5-80 Hz | Analysis range |
| Number of freqs | 100 | Frequency grid resolution |
| Wavelet | cmor1.5-1.0 | Complex Morlet |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb pywt


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel P4 (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Apply the wavelet transform

We use `pywt.cwt` with the complex Morlet wavelet. We convert frequencies to scales via `pywt.frequency2scale`, then compute the transform coefficients and extract the absolute value.

We apply the clean_signal function to remove noise via 1-45 Hz bandpass and 50 Hz notch filter.

In [ ]:
from scipy.signal import butter, filtfilt, iirnotch
import pywt

def clean_signal(signal, fs=200, low=1.0, high=45.0, notch_freq=50.0):
    b_bp, a_bp = butter(4, [low / (fs / 2), high / (fs / 2)], btype='band')
    filtered = filtfilt(b_bp, a_bp, signal)
    b_notch, a_notch = iirnotch(notch_freq, 30.0, fs=fs)
    filtered = filtfilt(b_notch, a_notch, filtered)
    return filtered

channel_data = clean_signal(channel_data, fs=fs)

n_plot = min(5000, len(channel_data))
signal = channel_data[:n_plot]

freqs = np.linspace(0.5, 80, 100)
scales = pywt.frequency2scale('cmor1.5-1.0', freqs / fs)
coefficients, _ = pywt.cwt(signal, scales, 'cmor1.5-1.0')
cwt_magnitude = np.abs(coefficients)
print(f'CWT matrix shape: {cwt_magnitude.shape}')

## 5. Interactive plot

**What to look for:**

- The x-axis represents time, the y-axis represents frequency
- Light colors indicate high energy, dark colors indicate low energy
- Notice how frequency bands change over time (unlike Fourier transform)



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(n_plot) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Wavelet Scalogram (CWT) - Channel P4'))
fig.add_trace(go.Scatter(x=t_sec, y=signal, name='Signal',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
fig.add_trace(go.Heatmap(z=cwt_magnitude, x=t_sec, y=freqs,
                         colorscale='Viridis', name='Magnitude'),
              row=2, col=1)
fig.update_layout(height=800, title_text='Wavelet Transform (CWT) - Channel P4',
                  xaxis_title='Time (s)', xaxis2_title='Time (s)',
                  yaxis_title='Amplitude (uV)', yaxis2_title='Frequency (Hz)',
                  showlegend=False)
fig.show()


## What did we learn?

- The wavelet transform enables a time-frequency analysis that reveals how frequencies change over time
- The scalogram shows the energy at each frequency and each moment
- Unlike Fourier transform, this analysis identifies when each frequency appeared
- The Morlet wavelet combines temporal and frequency localization

